**Step 4 of 5.** Consumes: nothing (builds a small illustrative set-cover problem in this notebook). Produces: nothing persisted — this is a demonstration, not a data-generating step.

**Important:** this notebook reproduces the *method* behind the paper's Figure 4 (solving one Benders-iteration's minimum-set-cover QUBO with three different QAOA backends), not the exact published numbers. The archived Fermioniq/IBM-hardware baseline plotted in Figure 4 came from one-off runs against paid/private services and cannot be regenerated here (see notebook 05's header for details). The MPS-JuliQAOA backend below needs no special credentials and *can* be run for real; Qiskit defaults to a free local simulator; Fermioniq and IBM-hardware execution are optional and skipped automatically if you don't have credentials configured.

## Minimum set cover as a QUBO, solved with three QAOA backends
Section III of the paper frames the Benders cut-selection step as a minimum set cover problem (binary variable per candidate cut, one coverage constraint per element to cover), then solves it as a QUBO with QAOA. Below we build a small illustrative instance of that problem with `milp_engine.strategies.minimum_set_cover_strategy.MinimumSetCoverStrategy` (the same class the real pipeline uses inside `BenderMILPSolver`), then hand it to `milp_engine.solvers.quantum_solver.QuantumSolver` configured with each of the three backends used in the paper: **MPS-JuliQAOA** (tensor-network emulator), **Qiskit** (local simulator, or real IBM Quantum hardware), and **Fermioniq's Ava** (tensor-network emulator, paid cloud service).

In [ ]:
import numpy as np

from milp_engine.strategies.minimum_set_cover_strategy import MinimumSetCoverStrategy

# A small illustrative set-cover instance: 6 candidate cuts (rows), 5 elements to
# cover (columns). Random but seeded, with every column guaranteed at least one
# covering cut so the instance is always feasible.
rng = np.random.default_rng(42)
n_cuts, n_elements = 6, 5
binary_indicator_matrix = (rng.random((n_cuts, n_elements)) < 0.4).astype(int)
for j in range(n_elements):
    if binary_indicator_matrix[:, j].sum() == 0:
        binary_indicator_matrix[rng.integers(n_cuts), j] = 1

print("Binary indicator matrix (cuts x elements):")
print(binary_indicator_matrix)

problem, decision_vars = MinimumSetCoverStrategy().create_optimization_problem(
    binary_indicator_matrix
)
print(problem)

### Backend 1: MPS-JuliQAOA
No external credentials needed — just the same Julia + `julia/MPS_JuliQAOA` setup as step 5 (see `julia/README.md`).

In [ ]:
from milp_engine.solvers.quantum_solver import JuliaMPSBackend, QuantumSolver

julia_solver = QuantumSolver(
    backend=JuliaMPSBackend(julia_options={"optimizer": "cobyla", "maxiter": 250}),
    solver_options={"reps": 1},
)
julia_solved = julia_solver.solve_with_binary_indicator_matrix(
    problem.copy(), binary_indicator_matrix
)
print("MPS-JuliQAOA selected cuts:", [v.name for v in decision_vars if v.varValue == 1])

### Backend 2: Qiskit
Runs on a free local Aer simulator by default (`use_hardware=False`, no credentials needed). To reproduce the IBM-hardware bar from Figure 4 instead, set `use_hardware=True` and `token_path` to a file containing your own IBM Quantum API token — this will submit a real job to `ibm_strasbourg` (or another device you name) and may queue for a while.

In [ ]:
from milp_engine.solvers.quantum_solver import QiskitBackend

USE_IBM_HARDWARE = False  # flip to True (and set token_path) to use real hardware

qiskit_solver = QuantumSolver(
    backend=QiskitBackend(
        qiskit_options={
            "n_shots": 1000,
            "use_hardware": USE_IBM_HARDWARE,
            "token_path": "ibm_token.txt",  # only read if use_hardware=True
            "ibm_hardware_name": "ibm_strasbourg",
        }
    ),
    solver_options={"reps": 1},
)
qiskit_solved = qiskit_solver.solve_with_binary_indicator_matrix(
    problem.copy(), binary_indicator_matrix
)
print("Qiskit selected cuts:", [v.name for v in decision_vars if v.varValue == 1])

### Backend 3: Fermioniq (optional — needs a paid account)
`FermioniqBackend` requires a `tokens.json` file (with your own `id`/`secret`) in the current working directory; it raises `FileNotFoundError` at construction time otherwise. The cell below checks for that file first and skips gracefully if it's not there, rather than crashing the notebook.

In [ ]:
from pathlib import Path

if Path("tokens.json").exists():
    from milp_engine.solvers.quantum_solver import FermioniqBackend

    fermioniq_solver = QuantumSolver(
        backend=FermioniqBackend(fermioniq_options={"remote_config": "cpu-16", "n_shots": 1000}),
        solver_options={"reps": 1},
    )
    fermioniq_solved = fermioniq_solver.solve_with_binary_indicator_matrix(
        problem.copy(), binary_indicator_matrix
    )
    print("Fermioniq selected cuts:", [v.name for v in decision_vars if v.varValue == 1])
else:
    print("Skipping Fermioniq: no tokens.json found in the current directory.")

## Done
All three backends solve the same minimum-set-cover QUBO via `QuantumSolver.solve_with_binary_indicator_matrix` — the exact code path `BenderMILPSolver` uses for the cut-selection step in steps 4/5, just applied here to one illustrative instance instead of inside a full multi-iteration Benders loop.